9. Ground thruth exploration vs OSM
This notebooks explores the differences beteen the ground truth dats collected from official data repositories and the binary mask from OSM

In [ ]:
"""Collect metrics.csv and hparams.yaml from every
<root>/s{seed}/models/trained_models/version_{n}/ folder into one table."""
%load_ext autoreload
%autoreload 2

import os 
import sys
from pathlib import Path
 
import pandas as pd
import yaml

seed = 42 
# ---- CONFIG: adapt these to your logging keys ---------------------------------
 
 # Add the root directory to Python path so that we can import the modules

root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(root_dir)

print(f"ROOT: {root_dir}")
GLOB_tr = "s*/models/trained_models/version_*"
GLOB_ts = "s*/models/test_result/version_*"
 
# hparams.yaml keys (first one found wins; dotted paths reach into nested dicts)
HPARAM_KEYS = {
    "Input features": ["input_features", "features", "bands", "model.input_features"],
    "Loss function": ["loss_function", "loss", "criterion", "model.loss"],
}
feat_versions = {"version_0": "S", "version_1": "BM", "version_2": "SDT", "version_3": "S", "version_4": "BM", "version_5": "SDT", "version_6": "S", "version_7": "BM",
           "version_8": "SDT", "version_9": "S", "version_10": "BM", "version_11": "SDT", "version_12": "S", "version_13": "BM", "version_14": "SDT"}
 
# metrics.csv columns (first one found wins)
METRIC_KEYS = {
    "IoU": ["IoU", "iou"],
    "Precision": ["Precision",  "precision"],
    "Recall": ["Recall",  "recall"],
    "F1 score": ["F1 Score", "f1_score", "F1", "f1"],
    "Accuracy": ["Accuracy"],
}
# -------------------------------------------------------------------------------


def dig(d, dotted):
    """Look up 'a.b.c' in a nested dict; return None if missing."""
    for part in dotted.split("."):
        if not isinstance(d, dict) or part not in d:
            return None
        d = d[part]
    return d
 
 
def first_hit(d, keys):
    for k in keys:
        v = dig(d, k)
        if v is not None:
            return ", ".join(map(str, v)) if isinstance(v, (list, tuple)) else v
    return None
 
 
def last_valid(df, keys):
    """Last non-NaN value of the first matching column (Lightning CSV logs are sparse)."""
    for k in keys:
        if k in df.columns and df[k].notna().any():
            return df[k].dropna().iloc[-1]
    return None
 
 
def natural_key(p: Path):
    seed = p.parents[2].name  # s{seed}
    ver = p.name.split("_")[-1]
    s = int(seed[1:]) if seed[1:].isdigit() else 0
    v = int(ver) if ver.isdigit() else 0
    return s, v
 
 
rows_tr = []
for vdir in sorted(Path(root_dir).glob(GLOB_tr), key=natural_key):
    hp_file = vdir / "hparams.yaml"
    if not hp_file.exists():
        print(f"skipping {vdir} (missing hparams.yaml)")
        continue
 
    hparams = yaml.safe_load(hp_file.read_text()) or {}
    
 
    row = {"Experiment": f"{vdir.parents[2].name}/{vdir.name}"}
    for col, keys in HPARAM_KEYS.items():
        row[col] = first_hit(hparams, keys)
    rows_tr.append(row)

# == get the test metrics from the test_result folders (if they exist) ==
rows_ts = []
for vdir in sorted(Path(root_dir).glob(GLOB_ts), key=natural_key):
    m_file = vdir / "test_metrics.csv"
    if not (m_file.exists()):
        print(f"skipping {vdir}  (metrics.csv)")
        continue
 
    metrics = pd.read_csv(m_file)
 
    row = {"Experiment": f"{vdir.parents[2].name}/{vdir.name}","Input features": feat_versions.get(vdir.name, "Unknown")}
 
    for col, keys in METRIC_KEYS.items():
        row[col] = last_valid(metrics, keys)
    rows_ts.append(row)

# concatenate the training and test results, preferring test metrics if available
rows = {r["Experiment"]: r for r in rows_tr}
for r in rows_ts:
    if r["Experiment"] in rows:
        rows[r["Experiment"]].update(r)
    else:
        rows[r["Experiment"]] = r
table = pd.DataFrame(list(rows.values()))
table['Seed'] = table['Experiment'].apply(lambda x: int(x.split('/')[0][1:]) if x.split('/')[0][1:].isdigit() else 0)
table['Version'] = table['Experiment'].apply(lambda x: int(x.split('/')[1].split('_')[-1]) if x.split('/')[1].split('_')[-1].isdigit() else 0)
if table.empty:
    sys.exit(f"No runs found under {Path(root_dir).resolve()} with pattern {GLOB}")
 
metric_cols = list(METRIC_KEYS)
table[metric_cols] = table[metric_cols].astype(float).round(4)

print(table.sort_values(by=["Seed","IoU","F1 score"], ascending=False).to_string(index=False))
table.sort_values(by="F1 score", ascending=False).to_csv(Path(root_dir) / f"overview_results_table{seed}.csv", index=False)
print(f"\nSaved results_table.csv and results_table.xlsx in {Path(root_dir).resolve()}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Project root = first parent of the working dir that contains the pugs_detection package,
# so this cell works even if the setup cell above was not run.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pugs_detection").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pugs_detection.aggregate_seeds import aggregate, collect

SEEDS = None          # e.g. [1, 7, 10] to drop s42
VERSIONS = [0, 1]
CI = 0.95

RATE_METRICS = ["Precision", "Recall", "F1 Score", "Accuracy", "IoU"]
VERSION_COLORS = {0: "#2a78d6", 1: "#eb6834"}   # blue, orange (colorblind-safe pair)
INK, INK_MUTED, GRID = "#1f1f1e", "#6b6a63", "#e6e5df"


# ----------------------------------------------------------------------------
# 1) Results as DataFrames
# ----------------------------------------------------------------------------
def build_results(root=ROOT, seeds=SEEDS, versions=VERSIONS, ci=CI):
    """Return (runs, summary).

    runs    : one row per seed/version/metric(/area_group) with the raw value
    summary : one row per version/metric(/area_group) with n_seeds, mean, std, CI
    """
    tm = collect(root, "test_metrics.csv", seeds, versions)
    pr = collect(root, "per_instance_recall.csv", seeds, versions)
    group_order = list(dict.fromkeys(pr["area_group"]))

    pooled = pr.groupby(["seed", "version"])[["n", "n_detected"]].sum().reset_index()
    pooled["Instance recall (overall)"] = pooled["n_detected"] / pooled["n"]

    metric_cols = [c for c in tm.columns if c not in ("seed", "version")]
    parts = [
        ("pixel", aggregate(tm, ["version"], metric_cols, ci)),
        ("instance_by_area", aggregate(pr, ["version", "area_group"], ["recall"], ci)),
        ("instance_overall", aggregate(pooled, ["version"], ["Instance recall (overall)"], ci)),
    ]
    summary = pd.concat([df.assign(level=lvl) for lvl, df in parts], ignore_index=True)
    summary["n_seeds"] = summary["n_seeds"].astype(int)
    summary["area_group"] = pd.Categorical(summary.get("area_group"), categories=group_order, ordered=True)
    summary = summary[["level", "version", "area_group", "metric", "n_seeds",
                       "mean", "std", "ci_low", "ci_high", "ci_half_width", "seeds"]]

    runs = pd.concat([
        tm.melt(id_vars=["seed", "version"], value_vars=metric_cols,
                var_name="metric", value_name="value").assign(level="pixel"),
        pr.rename(columns={"recall": "value"})[["seed", "version", "area_group", "value"]]
          .assign(metric="recall", level="instance_by_area"),
        pooled.melt(id_vars=["seed", "version"], value_vars=["Instance recall (overall)"],
                    var_name="metric", value_name="value").assign(level="instance_overall"),
    ], ignore_index=True)
    runs["area_group"] = pd.Categorical(runs["area_group"], categories=group_order, ordered=True)
    return runs, summary


def paper_table(summary, decimals=3):
    """Wide table: rows = metric (/area group), columns = version, cells = 'mean ± std [lo, hi]'."""
    s = summary[~summary["metric"].isin(["True Negative", "False Positive",
                                          "False Negative", "True Positive"])].copy()
    f = f"{{:.{decimals}f}}"
    s["cell"] = [f"{f.format(m)} ± {f.format(sd)} [{f.format(lo)}, {f.format(hi)}]"
                 for m, sd, lo, hi in zip(s["mean"], s["std"], s["ci_low"], s["ci_high"])]
    s["row"] = np.where(s["level"] == "instance_by_area",
                        "Recall area " + s["area_group"].astype(str), s["metric"])
    order = list(dict.fromkeys(s["row"]))
    t = s.pivot(index="row", columns="version", values="cell").reindex(order)
    t.columns = [f"version {v}" for v in t.columns]
    return t


# ----------------------------------------------------------------------------
# 2) Plots: mean ± CI with every seed shown as a dot
# ----------------------------------------------------------------------------
def _style(ax):
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=INK_MUTED, length=0)
    ax.yaxis.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def _dot_ci(ax, summary, runs, x_col, x_order, versions):
    """Dodged mean marker + CI whisker per version, raw seeds as small dots."""
    width = 0.5
    offsets = np.linspace(-width / 2, width / 2, len(versions)) if len(versions) > 1 else [0]
    rng = np.random.default_rng(0)
    xpos = {k: i for i, k in enumerate(x_order)}
    for off, v in zip(offsets, versions):
        c = VERSION_COLORS.get(v, INK)
        s = summary[summary["version"] == v]
        x = s[x_col].map(xpos).astype(float) + off
        ax.errorbar(x, s["mean"], yerr=[s["mean"] - s["ci_low"], s["ci_high"] - s["mean"]],
                    fmt="o", ms=8, color=c, ecolor=c, elinewidth=2, capsize=0,
                    markeredgecolor="white", markeredgewidth=1.5, zorder=3,
                    label=f"version {v}")
        r = runs[runs["version"] == v]
        xr = r[x_col].map(xpos).astype(float) + off + 0.07 + rng.uniform(-0.02, 0.02, len(r))
        ax.scatter(xr, r["value"], s=14, color=c, alpha=0.35, linewidths=0, zorder=2)
    ax.set_xticks(range(len(x_order)), [str(k) for k in x_order], color=INK)


def plot_results(runs, summary, ci=CI, out=None):
    versions = sorted(summary["version"].unique())
    pct = int(round(ci * 100))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), gridspec_kw={"width_ratios": [1, 1.1]})

    # (a) Pixel-level metrics
    ax = axes[0]
    s = summary[(summary["level"] == "pixel") & summary["metric"].isin(RATE_METRICS)]
    r = runs[(runs["level"] == "pixel") & runs["metric"].isin(RATE_METRICS)]
    _dot_ci(ax, s, r, "metric", RATE_METRICS, versions)
    ax.set_title("(a) Pixel-level test metrics", loc="left", color=INK)
    ax.set_ylabel(f"Score (mean, {pct}% CI)", color=INK_MUTED)

    # (b) Instance recall by object area
    ax = axes[1]
    s = summary[summary["level"] == "instance_by_area"]
    r = runs[runs["level"] == "instance_by_area"]
    groups = list(s["area_group"].cat.categories)
    _dot_ci(ax, s, r, "area_group", groups, versions)
    ax.set_title("(b) Instance recall by object area", loc="left", color=INK)
    ax.set_xlabel("Area group", color=INK_MUTED)
    ax.set_ylabel(f"Recall (mean, {pct}% CI)", color=INK_MUTED)
    ax.set_ylim(0, 1.05)

    for ax in axes:
        _style(ax)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, frameon=False, loc="upper right", ncol=len(versions), labelcolor=INK)
    fig.text(0.01, 0.01, f"Dots = individual seeds; marker = mean; whisker = {pct}% t-interval "
             f"(n = {summary['n_seeds'].min()}-{summary['n_seeds'].max()} seeds).",
             color=INK_MUTED, fontsize=8)
    fig.tight_layout(rect=(0, 0.04, 1, 0.93))
    if out:
        fig.savefig(out, dpi=300, bbox_inches="tight")
    return fig

In [ ]:
runs, summary = build_results()
with pd.option_context("display.width", 250, "display.max_colwidth", 40):
    print(summary)
    print(paper_table(summary))
out_dir = ROOT / "aggregated_results"
out_dir.mkdir(exist_ok=True)
summary.to_csv(out_dir / "summary_all.csv", index=False)
plot_results(runs, summary, out=out_dir / "results.png")
plt.show()

In [ ]:
from scipy import stats


def summarize(values, ci):
    """Mean, sample std and t-based CI of the mean across seeds."""
    x = np.asarray(values, dtype=float)
    x = x[~np.isnan(x)]
    n = len(x)
    median = np.median(x) if n else np.nan
    std = x.std(ddof=1) if n > 1 else np.nan
    if n > 1:
        half = stats.t.ppf((1 + ci) / 2, df=n - 1) * std / np.sqrt(n)
    else:
        half = np.nan
    return pd.Series({
        "n_seeds": n,
        "median": median,
        "std": std,
        "ci_low": median - half,
        "ci_high": median + half,
        "ci_half_width": half,
    })


def main():
    root = ROOT
    versions = [0, 1]
    ci = 0.95
  
    out_dir = root / "aggregated_results"
    out_dir.mkdir(parents=True, exist_ok=True)
    pct = int(round(ci * 100))

    # --- Global test metrics ---
    tm = collect(root, "test_metrics.csv", None, versions)
    if tm.empty:
        print("No test_metrics.csv found.")
    else:
        metric_cols = [c for c in tm.columns if c not in ("seed", "version")]
        tm.to_csv(out_dir / "test_metrics_all_runs.csv", index=False)
        summ = aggregate(tm, ["version"], metric_cols, ci)
        summ.to_csv(out_dir / "test_metrics_summary.csv", index=False)
        print(f"\n=== Test metrics (mean +/- std, {pct}% CI) ===")
        print_table(summ, ["version", "metric"])

    # --- Per-instance recall per area group ---
    pr = collect(root, "per_instance_recall.csv", None, versions)
    if pr.empty:
        print("No per_instance_recall.csv found.")
    else:
        group_order = list(dict.fromkeys(pr["area_group"]))
        pr.to_csv(out_dir / "per_instance_recall_all_runs.csv", index=False)
        summ = aggregate(pr, ["version", "area_group"], ["recall", "n_detected", "n"], ci)
        summ["area_group"] = pd.Categorical(summ["area_group"], categories=group_order, ordered=True)
        summ = summ.sort_values(["version", "metric", "area_group"]).reset_index(drop=True)
        summ.to_csv(out_dir / "per_instance_recall_summary.csv", index=False)
        print(f"\n=== Per-instance recall by area group (mean +/- std, {pct}% CI) ===")
        print_table(summ[summ["metric"] == "recall"], ["version", "area_group"])

        # Overall instance recall per run (all groups pooled), then across seeds
        pooled = (pr.groupby(["seed", "version"])[["n", "n_detected"]].sum().reset_index())
        pooled["instance_recall_overall"] = pooled["n_detected"] / pooled["n"]
        summ = aggregate(pooled, ["version"], ["instance_recall_overall"], ci)
        summ.to_csv(out_dir / "instance_recall_overall_summary.csv", index=False)
        print(f"\n=== Overall instance recall (pooled over groups) ===")
        print_table(summ, ["version", "metric"])

    print(f"\nSaved results to: {out_dir}")


def print_table(summ, keys):
    view = summ[keys + ["n_seeds"]].astype({"n_seeds": int})
    view["median +/- std"] = [f"{m:.4f} +/- {s:.4f}" for m, s in zip(summ["median"], summ["std"])]
    view["CI"] = [f"[{lo:.4f}, {hi:.4f}]" for lo, hi in zip(summ["ci_low"], summ["ci_high"])]
    view["seeds"] = summ["seeds"]
    with pd.option_context("display.max_rows", None, "display.width", 200):
        print(view.to_string(index=False))


main()